# ✈️ Análisis de Datos de Aviación — AviationStack API
> **Proyecto de Portfolio** | Analista de Datos | Python · Pandas · Plotly

---

## 📋 Tabla de Contenidos
1. [Configuración y API](#configuracion)
2. [Extracción de Datos](#extraccion)
3. [Limpieza y Transformación](#limpieza)
4. [Análisis Exploratorio](#eda)
5. [Dashboards Interactivos](#dashboards)
6. [Conclusiones e Insights](#conclusiones)

---

### 🎯 Objetivo del Proyecto
Analizar datos de vuelos en tiempo real utilizando la API de AviationStack para identificar:
- **Patrones de puntualidad** por aerolínea y ruta
- **Distribución de estados** de vuelo (a tiempo, retrasado, cancelado)
- **Volumen de tráfico** por aeropuerto y franja horaria
- **Tendencias geográficas** en la actividad aérea mundial

### 🔑 Obtener API Key GRATUITA
> Regístrate en [aviationstack.com](https://aviationstack.com) — el plan gratuito incluye **100 solicitudes/mes** (suficiente para este proyecto)

---

## 1. Configuración del Entorno <a name='configuracion'></a>

In [23]:
# ─── Instalar dependencias ───────────────────────────────────────────────────
!pip install -q requests pandas plotly kaleido
!pip install -U kaleido

In [24]:
# ─── Importaciones principales ───────────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
from google.colab import userdata
userdata.get('API_KEY')

# ─── Tema global de Plotly ───────────────────────────────────────────────────
pio.templates.default = 'plotly_white'

COLORES = {
    'primario':   '#1E3A5F',
    'secundario': '#2980B9',
    'acento':     '#E74C3C',
    'exito':      '#27AE60',
    'advertencia':'#F39C12',
    'claro':      '#ECF0F1',
}

print('✅ Entorno listo!')
print(f'📅 Fecha de análisis: {datetime.now().strftime("%d/%m/%Y %H:%M")}')

✅ Entorno listo!
📅 Fecha de análisis: 14/05/2026 18:48


In [25]:
# ─── Configuración de la API ─────────────────────────────────────────────────
# ⚠️  Reemplazá con tu clave en aviationstack.com
API_KEY  = '5004c74ce91664db43fba110577536d1I'
BASE_URL = 'http://api.aviationstack.com/v1'

def llamar_api(endpoint: str, params: dict = {}) -> dict | None:
    """
    Función genérica para consultar la API de AviationStack.
    Retorna el JSON como dict, o None si hay error.
    """
    params['access_key'] = API_KEY
    url = f'{BASE_URL}/{endpoint}'
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.HTTPError as e:
        print(f'❌ Error HTTP: {e}')
    except requests.exceptions.Timeout:
        print('❌ Tiempo de espera agotado — intentá de nuevo')
    except Exception as e:
        print(f'❌ Error inesperado: {e}')
    return None

# ─── Verificar conexión ───────────────────────────────────────────────────────
prueba = llamar_api('flights', {'limit': 1})
if prueba and 'data' in prueba:
    print('✅ API conectada exitosamente!')
    print(f'📊 Registros disponibles: {prueba.get("pagination", {}).get("total", "N/A")}')
else:
    print('⚠️  Sin conexión — se cargará el DATASET DE DEMOSTRACIÓN')

❌ Error HTTP: 401 Client Error: Unauthorized for url: http://api.aviationstack.com/v1/flights?limit=1&access_key=5004c74ce91664db43fba110577536d1I
⚠️  Sin conexión — se cargará el DATASET DE DEMOSTRACIÓN


## 2. Extracción de Datos <a name='extraccion'></a>

In [26]:
# ─── Generador de datos de demostración ──────────────────────────────────────
def generar_vuelos_demo(n: int = 300) -> list:
    """Simula datos de vuelos realistas para demostración del portfolio."""
    np.random.seed(42)

    aerolineas = [
        ('American Airlines', 'AA'),
        ('Delta Air Lines',   'DL'),
        ('United Airlines',   'UA'),
        ('Southwest Airlines','WN'),
        ('Lufthansa',         'LH'),
        ('British Airways',   'BA'),
        ('Emirates',          'EK'),
        ('Air France',        'AF'),
        ('Ryanair',           'FR'),
        ('Aerolíneas Argentinas', 'AR'),
    ]

    aeropuertos = [
        ('JFK', 'Nueva York',      'EE.UU.',   40.6413, -73.7781),
        ('LAX', 'Los Ángeles',     'EE.UU.',   33.9425,-118.4081),
        ('LHR', 'Londres',         'RU',       51.4700,  -0.4543),
        ('CDG', 'París',           'Francia',  49.0097,   2.5479),
        ('DXB', 'Dubái',           'EAU',      25.2532,  55.3657),
        ('FRA', 'Fráncfort',       'Alemania', 50.0379,   8.5622),
        ('AMS', 'Ámsterdam',       'Países Bajos',52.3086, 4.7639),
        ('GRU', 'São Paulo',       'Brasil',  -23.4356, -46.4731),
        ('EZE', 'Buenos Aires',    'Argentina',-34.8222,-58.5358),
        ('SCL', 'Santiago',        'Chile',   -33.3930, -70.7858),
        ('MEX', 'Ciudad de México','México',   19.4363, -99.0721),
        ('MIA', 'Miami',           'EE.UU.',   25.7959, -80.2870),
        ('ORD', 'Chicago',         'EE.UU.',   41.9742, -87.9073),
        ('SIN', 'Singapur',        'Singapur',  1.3644, 103.9915),
        ('NRT', 'Tokio',           'Japón',    35.7720, 140.3929),
    ]

    estados  = ['programado','activo','aterrizado','demorado','cancelado','incidente']
    pesos    = [0.30, 0.25, 0.22, 0.15, 0.06, 0.02]

    vuelos = []
    for i in range(n):
        al   = aerolineas[np.random.randint(len(aerolineas))]
        dep  = aeropuertos[np.random.randint(len(aeropuertos))]
        arr  = aeropuertos[np.random.randint(len(aeropuertos))]
        while arr == dep:
            arr = aeropuertos[np.random.randint(len(aeropuertos))]

        hora_dep   = np.random.randint(0, 24)
        min_dep    = np.random.randint(0, 60)
        demora_min = int(np.random.exponential(20)) if np.random.rand() < 0.35 else 0
        duracion   = int(np.random.uniform(60, 720))

        hora_salida = datetime.now().replace(
            hour=hora_dep, minute=min_dep, second=0, microsecond=0
        ) + timedelta(days=np.random.randint(-7, 1))

        estado = np.random.choice(estados, p=pesos)
        if demora_min > 0 and estado == 'programado':
            estado = 'demorado'

        vuelos.append({
            'vuelo_iata':          f"{al[1]}{np.random.randint(100, 9999)}",
            'aerolinea_nombre':    al[0],
            'aerolinea_iata':      al[1],
            'salida_iata':         dep[0],
            'salida_ciudad':       dep[1],
            'salida_pais':         dep[2],
            'salida_lat':          dep[3],
            'salida_lon':          dep[4],
            'llegada_iata':        arr[0],
            'llegada_ciudad':      arr[1],
            'llegada_pais':        arr[2],
            'llegada_lat':         arr[3],
            'llegada_lon':         arr[4],
            'estado_vuelo':        estado,
            'salida_programada':   hora_salida,
            'demora_minutos':      demora_min,
            'duracion_vuelo_min':  duracion,
        })
    return vuelos

print('✅ Generador de datos de demostración listo!')

✅ Generador de datos de demostración listo!


In [27]:
# ─── Obtener datos de la API (o usar demo) ────────────────────────────────────
def obtener_vuelos(limite: int = 100) -> list:
    """Obtiene vuelos de la API. Usa datos demo si no hay conexión."""
    raw = llamar_api('flights', {'limit': limite, 'flight_status': 'active'})
    if not raw or 'data' not in raw or not raw['data']:
        print('⚠️  Usando dataset de demostración (300 vuelos sintéticos)')
        return generar_vuelos_demo(300)

    registros = []
    for f in raw['data']:
        dep = f.get('departure', {})
        arr = f.get('arrival', {})
        al  = f.get('airline', {})
        fl  = f.get('flight', {})
        registros.append({
            'vuelo_iata':         fl.get('iata', 'N/D'),
            'aerolinea_nombre':   al.get('name', 'Desconocida'),
            'aerolinea_iata':     al.get('iata', 'XX'),
            'salida_iata':        dep.get('iata', 'N/D'),
            'salida_ciudad':      dep.get('airport', 'N/D'),
            'salida_pais':        dep.get('country', 'N/D'),
            'salida_lat':         dep.get('lat', 0),
            'salida_lon':         dep.get('lon', 0),
            'llegada_iata':       arr.get('iata', 'N/D'),
            'llegada_ciudad':     arr.get('airport', 'N/D'),
            'llegada_pais':       arr.get('country', 'N/D'),
            'llegada_lat':        arr.get('lat', 0),
            'llegada_lon':        arr.get('lon', 0),
            'estado_vuelo':       f.get('flight_status', 'desconocido'),
            'salida_programada':  dep.get('scheduled'),
            'demora_minutos':     dep.get('delay', 0) or 0,
            'duracion_vuelo_min': 0,
        })
    print(f'✅ {len(registros)} vuelos cargados desde la API')
    return registros

vuelos_raw = obtener_vuelos(100)
print(f'📦 Dataset: {len(vuelos_raw)} vuelos cargados')

❌ Error HTTP: 401 Client Error: Unauthorized for url: http://api.aviationstack.com/v1/flights?limit=100&flight_status=active&access_key=5004c74ce91664db43fba110577536d1I
⚠️  Usando dataset de demostración (300 vuelos sintéticos)
📦 Dataset: 300 vuelos cargados


## 3. Limpieza y Transformación de Datos <a name='limpieza'></a>

In [28]:
import locale
# ─── Construir DataFrame ──────────────────────────────────────────────────────
df = pd.DataFrame(vuelos_raw)

# Parsear fechas
if df['salida_programada'].dtype == object:
    df['salida_programada'] = pd.to_datetime(df['salida_programada'], errors='coerce')

# Variables derivadas
df['hora']          = df['salida_programada'].dt.hour

# Define a mapping from English to Spanish day names
day_name_map = {
    'Monday':    'Lunes',
    'Tuesday':   'Martes',
    'Wednesday': 'Miércoles',
    'Thursday':  'Jueves',
    'Friday':    'Viernes',
    'Saturday':  'Sábado',
    'Sunday':    'Domingo'
}

# Get day names in English and then map to Spanish
df['dia_semana'] = df['salida_programada'].dt.day_name().map(day_name_map)

df['fecha']         = df['salida_programada'].dt.date
df['banda_demora']  = pd.cut(
    df['demora_minutos'],
    bins=[-1, 0, 15, 30, 60, 120, 9999],
    labels=['Sin demora', '1–15 min', '15–30 min', '30–60 min', '1–2 h', 'Más de 2 h']
)
df['esta_demorado'] = df['demora_minutos'] > 0
df['demora_larga']  = df['demora_minutos'] >= 60
df['ruta']          = df['salida_iata'] + ' → ' + df['llegada_iata']

# Etiquetas de estado en español
mapa_estado = {
    'scheduled':  'Programado',
    'programado': 'Programado',
    'active':     'En Vuelo',
    'activo':     'En Vuelo',
    'landed':     'Aterrizado',
    'aterrizado': 'Aterrizado',
    'delayed':    'Demorado',
    'demorado':   'Demorado',
    'cancelled':  'Cancelado',
    'cancelado':  'Cancelado',
    'incident':   'Incidente',
    'incidente':  'Incidente',
}
df['estado_label'] = df['estado_vuelo'].map(mapa_estado).fillna('Otro')

print('═' * 58)
print(f'  Registros totales   : {len(df):,}')
print(f'  Aerolíneas          : {df["aerolinea_nombre"].nunique()}')
print(f'  Aeropuertos salida  : {df["salida_iata"].nunique()}')
print(f'  Rutas únicas        : {df["ruta"].nunique()}')
print(f'  Demora promedio     : {df["demora_minutos"].mean():.1f} min')
print(f'  % Demorados         : {df["esta_demorado"].mean()*100:.1f}%')
print('═' * 58)

══════════════════════════════════════════════════════════
  Registros totales   : 300
  Aerolíneas          : 10
  Aeropuertos salida  : 15
  Rutas únicas        : 167
  Demora promedio     : 6.8 min
  % Demorados         : 31.7%
══════════════════════════════════════════════════════════


In [29]:
# ─── Reporte de calidad de datos ──────────────────────────────────────────────
print('📋 Reporte de Calidad de Datos')
print('─' * 42)
nulos_pct = (df.isnull().sum() / len(df) * 100).round(2)
for col, pct in nulos_pct[nulos_pct > 0].items():
    print(f'  {col:<35} {pct:>5}% nulos')
if nulos_pct.max() == 0:
    print('  ✅ Sin valores nulos detectados!')
print('─' * 42)
print(df.dtypes)

📋 Reporte de Calidad de Datos
──────────────────────────────────────────
  ✅ Sin valores nulos detectados!
──────────────────────────────────────────
vuelo_iata                    object
aerolinea_nombre              object
aerolinea_iata                object
salida_iata                   object
salida_ciudad                 object
salida_pais                   object
salida_lat                   float64
salida_lon                   float64
llegada_iata                  object
llegada_ciudad                object
llegada_pais                  object
llegada_lat                  float64
llegada_lon                  float64
estado_vuelo                  object
salida_programada     datetime64[ns]
demora_minutos                 int64
duracion_vuelo_min             int64
hora                           int32
dia_semana                    object
fecha                         object
banda_demora                category
esta_demorado                   bool
demora_larga                    bool

## 4. Análisis Exploratorio <a name='eda'></a>

In [30]:
# ─── Paleta de colores por estado ────────────────────────────────────────────
colores_estado = {
    'Programado': '#2980B9',
    'En Vuelo':   '#27AE60',
    'Aterrizado': '#8E44AD',
    'Demorado':   '#E67E22',
    'Cancelado':  '#E74C3C',
    'Incidente':  '#C0392B',
    'Otro':       '#95A5A6',
}

In [31]:
# ─── GRÁFICO 1: Distribución de Estados de Vuelo (Dona) ──────────────────────
conteo_estados = df['estado_label'].value_counts().reset_index()
conteo_estados.columns = ['Estado', 'Cantidad']
colores_grafico = [colores_estado.get(s, '#95A5A6') for s in conteo_estados['Estado']]

fig1 = go.Figure(go.Pie(
    labels=conteo_estados['Estado'],
    values=conteo_estados['Cantidad'],
    hole=0.55,
    marker_colors=colores_grafico,
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Vuelos: %{value}<br>Proporción: %{percent}<extra></extra>',
    pull=[0.05 if s in ['Cancelado', 'Incidente'] else 0 for s in conteo_estados['Estado']]
))
fig1.add_annotation(
    text=f"<b>{len(df)}</b><br>Vuelos<br>Totales",
    x=0.5, y=0.5, showarrow=False, font_size=16, align='center'
)
fig1.update_layout(
    title=dict(text='✈️  Distribución de Estados de Vuelo', font_size=18, x=0.02),
    legend=dict(orientation='v', x=1.0, y=0.5),
    height=420, margin=dict(t=60, b=20, l=20, r=20)
)
fig1.show()

In [32]:
# ─── GRÁFICO 2: Vuelos por Aerolínea (Barras Horizontales) ───────────────────
df_aerolinea = (
    df.groupby('aerolinea_nombre')
      .agg(
          total_vuelos=('vuelo_iata', 'count'),
          demora_prom=('demora_minutos', 'mean'),
          pct_demorado=('esta_demorado', 'mean')
      )
      .reset_index()
      .sort_values('total_vuelos', ascending=True)
)
df_aerolinea['demora_prom']   = df_aerolinea['demora_prom'].round(1)
df_aerolinea['pct_demorado']  = (df_aerolinea['pct_demorado'] * 100).round(1)

fig2 = go.Figure(go.Bar(
    x=df_aerolinea['total_vuelos'],
    y=df_aerolinea['aerolinea_nombre'],
    orientation='h',
    marker=dict(
        color=df_aerolinea['demora_prom'],
        colorscale='RdYlGn_r',
        colorbar=dict(title='Demora<br>prom. (min)', thickness=12),
        line=dict(color='white', width=0.5)
    ),
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Vuelos totales: %{x}<br>'
        'Demora promedio: %{marker.color:.1f} min'
        '<extra></extra>'
    )
))
fig2.update_layout(
    title=dict(text='🏢  Vuelos por Aerolínea (color = demora promedio)', font_size=18, x=0.02),
    xaxis_title='Cantidad de Vuelos',
    yaxis_title='',
    height=420,
    margin=dict(t=60, b=40, l=200, r=60)
)
fig2.show()

In [33]:
# ─── GRÁFICO 3: Tráfico Aéreo por Hora del Día ───────────────────────────────
df_hora = (
    df.dropna(subset=['hora'])
      .groupby(['hora', 'estado_label'])
      .size()
      .reset_index(name='cantidad')
)

fig3 = px.area(
    df_hora, x='hora', y='cantidad', color='estado_label',
    color_discrete_map=colores_estado,
    labels={'hora': 'Hora (UTC)', 'cantidad': 'Vuelos', 'estado_label': 'Estado'},
)
fig3.update_traces(
    mode='lines',
    hovertemplate='<b>%{fullData.name}</b><br>Hora: %{x}:00<br>Vuelos: %{y}<extra></extra>'
)
fig3.update_layout(
    title=dict(text='🕐  Tráfico Aéreo por Franja Horaria', font_size=18, x=0.02),
    xaxis=dict(tickmode='linear', tick0=0, dtick=2, ticksuffix=':00'),
    height=420, hovermode='x unified',
    margin=dict(t=60, b=40, l=60, r=20)
)
fig3.show()

In [34]:
# ─── GRÁFICO 4: Distribución de Demoras (Histograma) ─────────────────────────
df_demorados = df[df['demora_minutos'] > 0]

fig4 = px.histogram(
    df_demorados, x='demora_minutos',
    nbins=40,
    color_discrete_sequence=[COLORES['acento']],
    labels={'demora_minutos': 'Demora (minutos)', 'count': 'Vuelos'},
    marginal='box'
)
fig4.add_vline(
    x=df_demorados['demora_minutos'].median(),
    line_dash='dash', line_color=COLORES['primario'],
    annotation_text=f" Mediana: {df_demorados['demora_minutos'].median():.0f} min",
    annotation_position='top right'
)
fig4.update_layout(
    title=dict(text='⏱️  Distribución de Demoras (solo vuelos demorados)', font_size=18, x=0.02),
    height=420, bargap=0.05,
    margin=dict(t=60, b=40, l=60, r=20)
)
fig4.show()

In [35]:
# ─── GRÁFICO 5: Puntualidad por Aerolínea (Barras Agrupadas) ─────────────────
df_puntualidad = (
    df.groupby('aerolinea_nombre')
      .agg(
          a_tiempo=('esta_demorado', lambda x: (~x).sum()),
          demorados=('esta_demorado', 'sum'),
          total=('vuelo_iata', 'count')
      )
      .reset_index()
)
df_puntualidad['pct_a_tiempo'] = (
    df_puntualidad['a_tiempo'] / df_puntualidad['total'] * 100
).round(1)
df_puntualidad = df_puntualidad.sort_values('pct_a_tiempo', ascending=False)

fig5 = go.Figure()
fig5.add_trace(go.Bar(
    name='A tiempo',
    x=df_puntualidad['aerolinea_nombre'],
    y=df_puntualidad['a_tiempo'],
    marker_color=COLORES['exito'],
    hovertemplate='<b>%{x}</b><br>A tiempo: %{y}<extra></extra>'
))
fig5.add_trace(go.Bar(
    name='Demorado',
    x=df_puntualidad['aerolinea_nombre'],
    y=df_puntualidad['demorados'],
    marker_color=COLORES['acento'],
    hovertemplate='<b>%{x}</b><br>Demorados: %{y}<extra></extra>'
))
fig5.update_layout(
    barmode='group',
    title=dict(text='📊  Puntualidad vs Demoras por Aerolínea', font_size=18, x=0.02),
    xaxis_tickangle=-35,
    height=450,
    legend=dict(x=0.80, y=0.95),
    margin=dict(t=60, b=110, l=60, r=20)
)
fig5.show()

In [36]:
# ─── GRÁFICO 6: Mapa Mundial de Rutas Aéreas ─────────────────────────────────
muestra_rutas = df.dropna(
    subset=['salida_lat', 'salida_lon', 'llegada_lat', 'llegada_lon']
).head(80)

fig6 = go.Figure()

# Trazar rutas
for _, fila in muestra_rutas.iterrows():
    color = '#E74C3C' if fila['esta_demorado'] else '#2980B9'
    fig6.add_trace(go.Scattergeo(
        lon=[fila['salida_lon'], fila['llegada_lon']],
        lat=[fila['salida_lat'], fila['llegada_lat']],
        mode='lines',
        line=dict(width=0.8, color=color),
        opacity=0.45,
        showlegend=False,
        hoverinfo='skip'
    ))

# Puntos de salida
fig6.add_trace(go.Scattergeo(
    lon=muestra_rutas['salida_lon'],
    lat=muestra_rutas['salida_lat'],
    mode='markers',
    marker=dict(size=5, color=COLORES['primario'], opacity=0.8),
    name='Aeropuerto',
    hovertemplate='<b>%{text}</b><extra></extra>',
    text=muestra_rutas['salida_iata']
))

fig6.update_layout(
    title=dict(text='🗺️  Rutas Aéreas Mundiales  🔵 A tiempo  🔴 Demorado', font_size=18, x=0.02),
    geo=dict(
        showland=True,      landcolor='#1A1A2E',
        showocean=True,     oceancolor='#0D1B2A',
        showcoastlines=True, coastlinecolor='#334455',
        showcountries=True,  countrycolor='#334455',
        showframe=False,
        projection_type='natural earth'
    ),
    height=520,
    paper_bgcolor='#0D1B2A',
    font_color='white',
    margin=dict(t=60, b=0, l=0, r=0)
)
fig6.show()

In [37]:
# ─── GRÁFICO 7: Aeropuertos Principales (Treemap) ────────────────────────────
df_aeropuertos = (
    df.groupby(['salida_iata', 'salida_ciudad', 'salida_pais'])
      .agg(
          vuelos=('vuelo_iata', 'count'),
          demora_prom=('demora_minutos', 'mean')
      )
      .reset_index()
      .sort_values('vuelos', ascending=False)
      .head(20)
)

fig7 = px.treemap(
    df_aeropuertos,
    path=[px.Constant('Mundo'), 'salida_pais', 'salida_iata'],
    values='vuelos',
    color='demora_prom',
    color_continuous_scale='RdYlGn_r',
    color_continuous_midpoint=df_aeropuertos['demora_prom'].median(),
    hover_data={'salida_ciudad': True, 'demora_prom': ':.1f'},
    labels={'demora_prom': 'Demora prom. (min)', 'vuelos': 'Vuelos'}
)
fig7.update_traces(textinfo='label+value')
fig7.update_layout(
    title=dict(text='🌍  Top Aeropuertos de Salida (color = demora promedio)', font_size=18, x=0.02),
    height=480,
    margin=dict(t=60, b=10, l=10, r=10)
)
fig7.show()

In [38]:
# ─── GRÁFICO 8: Bandas de Demora por Aerolínea (Burbujas) ────────────────────
df_burbujas = (
    df.groupby(['aerolinea_nombre', 'banda_demora'])
      .size()
      .reset_index(name='cantidad')
)

fig8 = px.scatter(
    df_burbujas,
    x='aerolinea_nombre', y='banda_demora', size='cantidad',
    color='cantidad',
    color_continuous_scale='Blues',
    size_max=40,
    labels={
        'aerolinea_nombre': 'Aerolínea',
        'banda_demora':     'Banda de Demora',
        'cantidad':         'Vuelos'
    },
    hover_data={'cantidad': True}
)
fig8.update_layout(
    title=dict(text='🔵  Bandas de Demora por Aerolínea (tamaño = cantidad de vuelos)', font_size=18, x=0.02),
    xaxis_tickangle=-35,
    height=450,
    margin=dict(t=60, b=120, l=110, r=20)
)
fig8.show()

## 5. Dashboards Interactivos <a name='dashboards'></a>

In [39]:
# ─── GRÁFICO 9: Dashboard de KPIs ────────────────────────────────────────────
fig_kpi = make_subplots(
    rows=2, cols=3,
    specs=[
        [{'type':'indicator'},{'type':'indicator'},{'type':'indicator'}],
        [{'type':'indicator'},{'type':'indicator'},{'type':'indicator'}]
    ],
    vertical_spacing=0.12
)

kpis = [
    ('Total de Vuelos',       len(df),                                     None, '#2980B9'),
    ('Demora Promedio (min)', df['demora_minutos'].mean(),                  15,   '#E67E22'),
    ('% Demorados',          df['esta_demorado'].mean() * 100,             25,   '#E74C3C'),
    ('Aerolíneas Activas',   df['aerolinea_nombre'].nunique(),              None, '#27AE60'),
    ('Aeropuertos Origen',   df['salida_iata'].nunique(),                   None, '#8E44AD'),
    ('% A Tiempo',           (~df['esta_demorado']).mean() * 100,          75,   '#1ABC9C'),
]

posiciones = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]

for (titulo, valor, ref, color), (fila, col) in zip(kpis, posiciones):
    kwargs = dict(
        value=round(valor, 1),
        title={'text': titulo, 'font': {'size': 13}},
        number={'font': {'color': color, 'size': 30}},
        mode='number+delta' if ref else 'number',
    )
    if ref:
        kwargs['delta'] = {'reference': ref, 'relative': False}
    fig_kpi.add_trace(go.Indicator(**kwargs), row=fila, col=col)

fig_kpi.update_layout(
    title=dict(text='📈  Dashboard de Indicadores Clave (KPIs)', font_size=20, x=0.02),
    height=380,
    paper_bgcolor='#F8F9FA',
    margin=dict(t=70, b=20, l=20, r=20)
)
fig_kpi.show()

In [40]:
# ─── GRÁFICO 10: Sunburst — Estado > Aerolínea > Banda de Demora ─────────────
df_sunburst = (
    df.groupby(['estado_label', 'aerolinea_nombre', 'banda_demora'])
      .size()
      .reset_index(name='cantidad')
)

fig10 = px.sunburst(
    df_sunburst,
    path=['estado_label', 'aerolinea_nombre', 'banda_demora'],
    values='cantidad',
    color='estado_label',
    color_discrete_map=colores_estado,
    hover_data=['cantidad']
)
fig10.update_traces(
    textinfo='label+percent parent',
    hovertemplate='<b>%{label}</b><br>Vuelos: %{value}<br>%{percentParent:.1%} del segmento padre<extra></extra>'
)
fig10.update_layout(
    title=dict(text='☀️  Análisis Jerárquico: Estado → Aerolínea → Banda de Demora', font_size=18, x=0.02),
    height=580,
    margin=dict(t=60, b=10, l=10, r=10)
)
fig10.show()

## 6. Conclusiones e Insights <a name='conclusiones'></a>

In [41]:
# ─── Generar insights automáticamente ────────────────────────────────────────
mejor_aerolinea  = df_aerolinea.sort_values('demora_prom').iloc[0]
peor_aerolinea   = df_aerolinea.sort_values('demora_prom', ascending=False).iloc[0]
aeropuerto_top   = df_aeropuertos.sort_values('vuelos', ascending=False).iloc[0]
hora_pico        = df.dropna(subset=['hora'])['hora'].mode()[0]
tasa_cancelacion = (df['estado_vuelo'].isin(['cancelado','cancelled'])).mean() * 100

print('=' * 62)
print('  📊 INSIGHTS PRINCIPALES — Análisis de Aviación')
print('=' * 62)
print(f"""
✅ Puntualidad general   : {(~df['esta_demorado']).mean()*100:.1f}% de vuelos a tiempo
⏱  Demora promedio       : {df['demora_minutos'].mean():.1f} minutos
❌ Tasa de cancelación   : {tasa_cancelacion:.1f}%

🏆 Aerolínea más puntual : {mejor_aerolinea['aerolinea_nombre']}
   → Demora prom.: {mejor_aerolinea['demora_prom']:.1f} min

⚠️  Aerolínea más demorada: {peor_aerolinea['aerolinea_nombre']}
   → Demora prom.: {peor_aerolinea['demora_prom']:.1f} min

🛫 Aeropuerto más activo  : {aeropuerto_top['salida_iata']}
   ({aeropuerto_top['salida_ciudad']}) — {aeropuerto_top['vuelos']} vuelos

🕐 Hora pico de salidas   : {hora_pico:02d}:00 UTC

📌 Distribución por banda de demora:
{df['banda_demora'].value_counts().to_string()}
""")
print('=' * 62)

  📊 INSIGHTS PRINCIPALES — Análisis de Aviación

✅ Puntualidad general   : 68.3% de vuelos a tiempo
⏱  Demora promedio       : 6.8 minutos
❌ Tasa de cancelación   : 3.7%

🏆 Aerolínea más puntual : United Airlines
   → Demora prom.: 1.9 min

⚠️  Aerolínea más demorada: British Airways
   → Demora prom.: 11.2 min

🛫 Aeropuerto más activo  : GRU
   (São Paulo) — 30 vuelos

🕐 Hora pico de salidas   : 16:00 UTC

📌 Distribución por banda de demora:
banda_demora
Sin demora    205
1–15 min       51
30–60 min      22
15–30 min      19
1–2 h           3
Más de 2 h      0



In [42]:
# ─── Exportar gráficos como HTML interactivo ──────────────────────────────────
from pathlib import Path

carpeta_salida = Path('graficos_aviacion')
carpeta_salida.mkdir(exist_ok=True)

graficos = {
    '01_distribucion_estados':   fig1,
    '02_vuelos_por_aerolinea':   fig2,
    '03_trafico_por_hora':       fig3,
    '04_histograma_demoras':     fig4,
    '05_puntualidad':            fig5,
    '06_mapa_rutas_mundial':     fig6,
    '07_treemap_aeropuertos':    fig7,
    '08_burbujas_demoras':       fig8,
    '09_dashboard_kpis':         fig_kpi,
    '10_sunburst_jerarquico':    fig10,
}

for nombre, fig in graficos.items():
    ruta = carpeta_salida / f'{nombre}.html'
    fig.write_html(str(ruta), include_plotlyjs='cdn')
    print(f'  ✅ Guardado: {ruta}')

print(f'\n📁 Todos los gráficos exportados en: ./{carpeta_salida}/')
print('   → Abrí cualquier archivo .html en tu navegador para la versión interactiva')

  ✅ Guardado: graficos_aviacion/01_distribucion_estados.html
  ✅ Guardado: graficos_aviacion/02_vuelos_por_aerolinea.html
  ✅ Guardado: graficos_aviacion/03_trafico_por_hora.html
  ✅ Guardado: graficos_aviacion/04_histograma_demoras.html
  ✅ Guardado: graficos_aviacion/05_puntualidad.html
  ✅ Guardado: graficos_aviacion/06_mapa_rutas_mundial.html
  ✅ Guardado: graficos_aviacion/07_treemap_aeropuertos.html
  ✅ Guardado: graficos_aviacion/08_burbujas_demoras.html
  ✅ Guardado: graficos_aviacion/09_dashboard_kpis.html
  ✅ Guardado: graficos_aviacion/10_sunburst_jerarquico.html

📁 Todos los gráficos exportados en: ./graficos_aviacion/
   → Abrí cualquier archivo .html en tu navegador para la versión interactiva
